# 🎬 AI-Powered Movie Recommendation System

**Beginner-friendly Content-Based Recommendation System**

This project uses the Kaggle **TMDB 5000 Movie Dataset** to recommend movies similar to a movie selected by the user.

### ML techniques used
- Data cleaning and preprocessing
- JSON metadata extraction
- Feature engineering
- NLP text vectorization with CountVectorizer
- Cosine similarity
- Content-based recommendation

### Project flow
`Raw movie data → Clean/parse metadata → Create tags → Vectorize → Cosine similarity → Top recommendations`

> Dataset source: Kaggle — TMDB 5000 Movie Dataset.


In [2]:
# 1. Import libraries
import ast
import pickle
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Load the Kaggle datasets

Download `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` from Kaggle and place them in `data/`.

The dataset contains about 5,000 movies and includes movie metadata plus cast/crew information.


In [3]:
movies = pd.read_csv("/content/tmdb_5000_credits.csv")
credits = pd.read_csv("/content/tmdb_5000_movies.csv")

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)

movies.head()


Movies shape: (4803, 4)
Credits shape: (4803, 20)


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [4]:
credits.head()


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [5]:
movies = movies.rename(columns={"movie_id": "id"})

df = movies.merge(credits, on="id", how="inner")

print("Merged shape:", df.shape)
df[["id", "title_x", "overview"]].head()

Merged shape: (4803, 23)


,id,title_x,overview
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [6]:
# 4. Rename title column and keep useful features
df = df.rename(columns={"title_x": "title"})

features = [
    "id", "title", "overview", "genres",
    "keywords", "cast", "crew",
    "vote_average", "vote_count", "popularity"
]

df = df[features].copy()

# Remove rows without essential recommendation information
df = df.dropna(subset=["title", "overview"])

print("Rows after cleaning:", len(df))
df.head()


Rows after cleaning: 4800


,id,title,overview,genres,keywords,cast,crew,vote_average,vote_count,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,11800,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,4500,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,4466,107.376788
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,9106,112.312950
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,2124,43.926995


## 5. Understand the JSON-like columns

Columns such as `genres`, `keywords`, `cast`, and `crew` contain lists stored as text.

Example:

`[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]`

We will extract only the useful names.


In [7]:
def parse_names(text):
    """Extract the 'name' field from a JSON-like list stored as text."""
    try:
        items = ast.literal_eval(text)
        return [item["name"].replace(" ", "").lower()
                for item in items if "name" in item]
    except (ValueError, SyntaxError, TypeError):
        return []

def parse_cast(text, n=3):
    try:
        items = ast.literal_eval(text)
        return [item["name"].replace(" ", "").lower()
                for item in items[:n] if "name" in item]
    except (ValueError, SyntaxError, TypeError):
        return []

def parse_director(text):
    try:
        items = ast.literal_eval(text)
        for item in items:
            if item.get("job") == "Director":
                return [item["name"].replace(" ", "").lower()]
        return []
    except (ValueError, SyntaxError, TypeError):
        return []


In [8]:
# 6. Create clean metadata features
df["genres"] = df["genres"].apply(parse_names)
df["keywords"] = df["keywords"].apply(parse_names)
df["cast"] = df["cast"].apply(parse_cast)
df["crew"] = df["crew"].apply(parse_director)

# Convert overview to individual lowercase words
df["overview"] = (
    df["overview"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9 ]", " ", regex=True)
)

df.head()


,id,title,overview,genres,keywords,cast,crew,vote_average,vote_count,popularity
0,19995,Avatar,in the 22nd century a paraplegic marine is di...,"[action, adventure, fantasy, sciencefiction]","[cultureclash, future, spacewar, spacecolony, ...","[samworthington, zoesaldana, sigourneyweaver]",[jamescameron],7.2,11800,150.437577
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believed to be dead ha...,"[adventure, fantasy, action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[johnnydepp, orlandobloom, keiraknightley]",[goreverbinski],6.9,4500,139.082615
2,206647,Spectre,a cryptic message from bond s past sends him o...,"[action, adventure, crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[danielcraig, christophwaltz, léaseydoux]",[sammendes],6.3,4466,107.376788
3,49026,The Dark Knight Rises,following the death of district attorney harve...,"[action, crime, drama, thriller]","[dccomics, crimefighter, terrorist, secretiden...","[christianbale, michaelcaine, garyoldman]",[christophernolan],7.6,9106,112.312950
4,49529,John Carter,john carter is a war weary former military ca...,"[action, adventure, sciencefiction]","[basedonnovel, mars, medallion, spacetravel, p...","[taylorkitsch, lynncollins, samanthamorton]",[andrewstanton],6.1,2124,43.926995


## 7. Build the `tags` feature

The recommendation engine needs one combined text field describing each movie.

We combine:
- overview
- genres
- keywords
- top 3 cast members
- director

This lets the model compare movies using their content.


In [9]:
df["tags"] = (
    df["overview"] + " " +
    df["genres"].apply(lambda x: " ".join(x)) + " " +
    df["keywords"].apply(lambda x: " ".join(x)) + " " +
    df["cast"].apply(lambda x: " ".join(x)) + " " +
    df["crew"].apply(lambda x: " ".join(x))
)

df["tags"] = df["tags"].str.replace(r"\s+", " ", regex=True).str.strip()

df[["title", "tags"]].head()


,title,tags
0,Avatar,in the 22nd century a paraplegic marine is dis...
1,Pirates of the Caribbean: At World's End,captain barbossa long believed to be dead has ...
2,Spectre,a cryptic message from bond s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,john carter is a war weary former military cap...


## 8. Convert movie text into numbers

Computers cannot directly compare sentences.

`CountVectorizer` converts words into numerical vectors.

We use at most 5,000 vocabulary terms and remove common English stop words.


In [10]:
cv = CountVectorizer(max_features=5000, stop_words="english")

vectors = cv.fit_transform(df["tags"])

print("Vector matrix shape:", vectors.shape)


Vector matrix shape: (4800, 5000)


## 9. Calculate cosine similarity

Cosine similarity measures how close two movie vectors are.

- Close to `1` → very similar
- Close to `0` → not very similar

The result is a movie-by-movie similarity matrix.


In [11]:
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)


Similarity matrix shape: (4800, 4800)


## 10. Recommendation function

The function:
1. Finds the selected movie's row.
2. Gets its similarity score against every movie.
3. Sorts movies from most similar to least similar.
4. Skips the selected movie itself.
5. Returns the top 5 recommendations.


In [12]:
def recommend(movie, n=5):
    matches = df[df["title"].str.lower() == movie.lower()]

    if matches.empty:
        return [f"Movie '{movie}' was not found in the dataset."]

    movie_index = matches.index[0]
    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:n+1]

    recommendations = []
    for index, score in movies_list:
        recommendations.append({
            "title": df.iloc[index]["title"],
            "similarity_score": round(float(score), 4)
        })

    return recommendations


In [13]:
# 11. Test the recommender
recommend("Aliens vs Predator")


["Movie 'Aliens vs Predator' was not found in the dataset."]

## 12. Save model artifacts

The Streamlit app does not need to retrain the model every time. We save the processed movie table and similarity matrix.


In [17]:
import os

# Define the directory path
output_dir = "model"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save artifacts one directory above this notebook's location
with open(os.path.join(output_dir, "movies.pkl"), "wb") as f:
    pickle.dump(df, f)

with open(os.path.join(output_dir, "similarity.pkl"), "wb") as f:
    pickle.dump(similarity, f)

with open(os.path.join(output_dir, "vectorizer.pkl"), "wb") as f:
    pickle.dump(cv, f)

print("Model artifacts saved.")

Model artifacts saved.


## 13. Simple model sanity check

A recommendation system is not a normal supervised ML problem, so there is no accuracy score like classification accuracy.

For this beginner project, we perform a sanity check:
- verify the requested movie exists
- verify recommendations are not the same movie
- inspect similarity scores


In [ ]:
test_movie = "Titan A.E."
results = recommend(test_movie, n=5)

print(f"Recommendations for: {test_movie}")
for item in results:
    print(f"- {item['title']} | similarity={item['similarity_score']}")


## 14. Load and Test Saved Models

Now, let's load the saved `movies.pkl`, `similarity.pkl`, and `vectorizer.pkl` files to ensure they can be used to make recommendations without re-running the entire notebook.

In [18]:
import pickle
import os

# Define the directory path where artifacts were saved
output_dir = "model"

# Load the saved artifacts
with open(os.path.join(output_dir, "movies.pkl"), "rb") as f:
    loaded_df = pickle.load(f)

with open(os.path.join(output_dir, "similarity.pkl"), "rb") as f:
    loaded_similarity = pickle.load(f)

with open(os.path.join(output_dir, "vectorizer.pkl"), "rb") as f:
    loaded_cv = pickle.load(f)

print("Model artifacts loaded successfully.")

# Re-define the recommend function to use the loaded artifacts
def loaded_recommend(movie, n=5):
    matches = loaded_df[loaded_df["title"].str.lower() == movie.lower()]

    if matches.empty:
        return [f"Movie '{movie}' was not found in the dataset."]

    movie_index = matches.index[0]
    distances = loaded_similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:n+1]

    recommendations = []
    for index, score in movies_list:
        recommendations.append({
            "title": loaded_df.iloc[index]["title"],
            "similarity_score": round(float(score), 4)
        })

    return recommendations

# Test the loaded recommender
test_movie_loaded = "Avatar"
loaded_results = loaded_recommend(test_movie_loaded, n=5)

print(f"\nRecommendations for: {test_movie_loaded} (using loaded model):")
for item in loaded_results:
    print(f"- {item['title']} | similarity={{item['similarity_score']}}")

Model artifacts loaded successfully.

Recommendations for: Avatar (using loaded model):
- Titan A.E. | similarity={item['similarity_score']}
- Small Soldiers | similarity={item['similarity_score']}
- Ender's Game | similarity={item['similarity_score']}
- Independence Day | similarity={item['similarity_score']}
- Aliens vs Predator: Requiem | similarity={item['similarity_score']}
